In [ ]:
# Fix SSL certificate trust using the certifi CA bundle (must run before other imports)
# Avoid disabling TLS verification globally; configure trusted CAs instead
import os
import certifi
os.environ["SSL_CERT_FILE"] = certifi.where()
os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()

In [ ]:
# Imports and shared configuration for the sentence-transformer WSD pipeline
from sentence_transformers import SentenceTransformer
from simple_wsd import process_senses_with_simple_wsd
from data_loader import load_sense_repo_by_round
from config import ANNOTATION_CHUNKS, DATA_DIR, OUTPUT_DIR, SIMPLE_WSD_MODEL_PRESETS
from writers import CustomWebAnnoTSVWriter, InceptionWebAnnoTSVWriter

# Select one embedding baseline: "simple" | "tesla" | "mling"
SIMPLE_WSD_MODEL_TAG = "mling"
if SIMPLE_WSD_MODEL_TAG not in SIMPLE_WSD_MODEL_PRESETS:
    raise ValueError(f"Unknown SIMPLE_WSD_MODEL_TAG={SIMPLE_WSD_MODEL_TAG!r}. Choose one of {sorted(SIMPLE_WSD_MODEL_PRESETS)}.")
MODEL_CONFIG = SIMPLE_WSD_MODEL_PRESETS[SIMPLE_WSD_MODEL_TAG]
MODEL_NAME = MODEL_CONFIG["model_name"]
ORIGIN_WSD = MODEL_CONFIG["origin"]
TEXT_PREFIX = MODEL_CONFIG.get("text_prefix", "")
NORMALIZE_EMBEDDINGS = MODEL_CONFIG.get("normalize_embeddings", False)

print(f"SimpleWSD tag  : {SIMPLE_WSD_MODEL_TAG}")
print(f"Origin tag     : {ORIGIN_WSD}")
print(f"Embedding model: {MODEL_NAME}")

# Annotation round to use (1 = old/first round, 2 = current/second round)
ROUND = 1

In [ ]:
# 1) Instantiate the sentence-transformer model
wsd_model = SentenceTransformer(MODEL_NAME)
print(f"Loaded model: {MODEL_NAME}")
print(f"Text prefix: {TEXT_PREFIX!r}; normalize embeddings: {NORMALIZE_EMBEDDINGS}")

In [ ]:
# 2) Load sense repository and select corpus chunk
senses_df = load_sense_repo_by_round(round_number=ROUND)
print(f"Loaded sense repo for round {ROUND}: {len(senses_df)} senses")

from webanno_spacy_converter.parsers.tsv_parser_v3 import WebAnnoLEXISParser

chunks = [(b, e, (DATA_DIR / fname)) for (b, e, fname) in ANNOTATION_CHUNKS]
print("Available chunks (index, begin, end, file):",
      [(i, b, e, p.name) for i, (b, e, p) in enumerate(chunks)])

chunk_idx = 0  # Select chunk index to process
chunk_begin, chunk_end, selected_path = chunks[chunk_idx]
print(f"Using chunk #{chunk_idx}: {selected_path.name} -> ({chunk_begin}, {chunk_end})")

parser = WebAnnoLEXISParser(selected_path)
sentences = parser.parse()

begin, end = chunk_begin, chunk_end

test = False
tb, te = 0, 100  # 0-based slice within selected chunk when test=True
if test:
    sentences = sentences[tb:te]
    begin, end = chunk_begin + tb, chunk_begin + te - 1

In [ ]:
# 3) Annotate sentences with cosine-similarity WSD
sentences = process_senses_with_simple_wsd(
    sentences,
    senses_df,
    model=wsd_model,
    origin_model=ORIGIN_WSD,
    progress=True,
    text_prefix=TEXT_PREFIX,
    normalize_embeddings=NORMALIZE_EMBEDDINGS,
)

In [ ]:
# 4) Persist annotated corpus
ROUND_SUFFIX = f"_round{ROUND}"
writer = CustomWebAnnoTSVWriter(sentences)
writer.save(OUTPUT_DIR / f"LexiSense_{begin:04d}_{end:04d}_{ORIGIN_WSD}{ROUND_SUFFIX}.tsv")

incept_writer = InceptionWebAnnoTSVWriter(sentences)
incept_writer.save(OUTPUT_DIR / f"LexiSense_Inception_{begin:04d}_{end:04d}_{ORIGIN_WSD}{ROUND_SUFFIX}.tsv")

# SimpleWSD_sense.ipynb

This notebook mirrors `ChatGPT_sense.ipynb` but relies on the local cosine-similarity pipeline (`process_senses_with_simple_wsd`) powered by sentence-transformer embeddings. Set `SIMPLE_WSD_MODEL_TAG` to `"simple"`, `"tesla"`, or `"mling"` to switch the baseline model, then adjust `chunk_idx` or the `test` slice to process different corpora.